# EDA Pipeline — Notebook de ejemplo

Este notebook ejecuta el pipeline de EDA a través de su API de Python (sin pasar por la CLI) sobre `data/raw/ecommerce.csv`, y muestra un resumen de los resultados.

Para el uso normal desde línea de comandos, vea `README.md` (comandos `analyze-file` / `analyze-batch`).

In [ ]:
import json
from pathlib import Path

from eda_pipeline.config import Config
from eda_pipeline.pipeline import EDAPipeline

## Configurar y ejecutar el pipeline

`data/raw/ecommerce.csv` incluye una columna target binaria `is_churn`; el resto de las opciones (delimitador, codificación, separador decimal) se auto-detectan.

In [ ]:
data_path = Path("..") / "data" / "raw" / "ecommerce.csv"

config = Config(
    input_file=str(data_path),
    output_dir=str(Path("..") / "reports"),
)
config.target.target_column = "is_churn"

pipeline = EDAPipeline(config)
pipeline.setup()
results = pipeline.run()

list(results.keys())

## Resumen de resultados

In [ ]:
dataset_name, result = next(iter(results.items()))

print(f"Dataset: {dataset_name}")
print(f"Éxito: {result['success']}")
print(f"Alertas: {result['n_alerts']} | Gráficos: {result['n_plots']}")
print(f"Reporte HTML: {result['html_report']}")
print(f"Resumen JSON: {result['summary_json']}")
if result.get("failed_steps"):
    print(f"Pasos con error: {result['failed_steps']}")

In [ ]:
summary = json.loads(Path(result["summary_json"]).read_text(encoding="utf-8"))

print("Tipos de columnas inferidos:")
for col, dtype in summary["column_types"].items():
    print(f"  {col}: {dtype}")

print("\nAlertas principales:")
for alert in summary["alerts"][:5]:
    print(f"  [{alert['severity'].upper()}] {alert['message']}")

if "target" in summary:
    print("\nAnálisis de target:")
    print(f"  Columna: {summary['target']['target_column']} ({summary['target']['target_type']})")
    class_balance = summary["target"]["class_balance"]
    if class_balance:
        print(f"  Desbalanceado: {class_balance['is_imbalanced']} (conteos: {class_balance['counts']})")

## Ver el reporte HTML

El reporte completo (auto-contenido, con todos los gráficos, tablas y alertas) está disponible en la ruta impresa arriba (`result["html_report"]`); ábralo directamente en el navegador.